# 🏸 Shuttlecock Detector — YOLOv8 Training Notebook (NEW)

**Athlete Vision Project** — Train a custom YOLOv8 model to detect shuttlecocks in badminton videos.

### ⚡ Quick Start:
1. Make sure you're on **GPU runtime** (Runtime → Change Runtime Type → T4 GPU)
2. Click **Runtime → Run All**
3. Login to Roboflow when prompted in Step 2
4. Download the trained `shuttlecock_best.pt` file at the end (~6MB)

### ⏱️ Total time: ~20-30 minutes

## Step 1: Install Dependencies

In [ ]:
# Install required packages
!pip install -q ultralytics roboflow

# Verify GPU is available
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️ WARNING: No GPU detected! Go to Runtime → Change Runtime Type → T4 GPU")

## Step 2: 🔑 Authenticate with Roboflow

Running the cell below will print a link. Click the link to log in to your Roboflow account, copy the authentication token, and paste it back here.

In [ ]:
import roboflow
# Log in interactively
roboflow.login()

## Step 3: Download Shuttlecock Dataset from Roboflow

We download the correct Mathieu Cartron shuttlecock dataset version containing 8k images.

In [ ]:
from roboflow import Roboflow
import os

# Initialize Roboflow using the authenticated session
rf = Roboflow()

# List of popular shuttlecock datasets to try (ordered by quality)
# We try multiple versions (v1, v2, v3) of the Mathieu Cartron project
datasets_to_try = [
    ("mathieu-cartron", "shuttlecock-cqzy3", 1, "Shuttlecock by Mathieu Cartron — 8k images (v1)"),
    ("mathieu-cartron", "shuttlecock-cqzy3", 3, "Shuttlecock by Mathieu Cartron — 8k images (v3)"),
    ("mathieu-cartron", "shuttlecock-cqzy3", 2, "Shuttlecock by Mathieu Cartron — 8k images (v2)"),
]

dataset = None
dataset_location = None

for workspace, project_name, version_num, desc in datasets_to_try:
    try:
        print(f"\n🔄 Trying: {desc}")
        print(f"   → {workspace}/{project_name}/v{version_num}")
        project = rf.workspace(workspace).project(project_name)
        version = project.version(version_num)
        dataset = version.download("yolov8", location=f"./shuttlecock_dataset")
        dataset_location = "./shuttlecock_dataset"
        print(f"✅ SUCCESS! Downloaded: {desc}")
        break
    except Exception as e:
        print(f"   ❌ Failed: {str(e)[:150]}")
        continue

if dataset is None:
    print("\n⚠️ None of the preset datasets worked.")
else:
    # Show dataset contents
    print(f"\n📁 Dataset downloaded to: {dataset_location}")
    for root, dirs, files in os.walk(dataset_location):
        level = root.replace(dataset_location, '').count(os.sep)
        indent = ' ' * 2 * level
        print(f'{indent}{os.path.basename(root)}/')
        if level < 2:
            subindent = ' ' * 2 * (level + 1)
            img_count = len([f for f in files if f.endswith(('.jpg','.png','.jpeg'))])
            label_count = len([f for f in files if f.endswith('.txt')])
            other = len(files) - img_count - label_count
            if img_count: print(f'{subindent}📷 {img_count} images')
            if label_count: print(f'{subindent}🏷️ {label_count} labels')
            if other: print(f'{subindent}📄 {other} other files')

## Step 4: Fix data.yaml Paths
Ensure the dataset YAML file has correct absolute paths for Colab.

In [ ]:
import yaml
import os

yaml_path = os.path.join(dataset_location, "data.yaml")

# Read existing yaml
with open(yaml_path, 'r') as f:
    data = yaml.safe_load(f)

print("Original data.yaml:")
print(data)

# Fix paths to be absolute
abs_dataset = os.path.abspath(dataset_location)
data['train'] = os.path.join(abs_dataset, 'train', 'images')
data['val'] = os.path.join(abs_dataset, 'valid', 'images')
if 'test' in data:
    data['test'] = os.path.join(abs_dataset, 'test', 'images')

# Write fixed yaml
with open(yaml_path, 'w') as f:
    yaml.dump(data, f, default_flow_style=False)

print(f"\n✅ Fixed data.yaml:")
print(f"  Train: {data['train']}")
print(f"  Valid: {data['val']}")
print(f"  Classes: {data.get('names', data.get('nc', 'unknown'))}")

## Step 5: 🚀 Train YOLOv8 Shuttlecock Detector

We fine-tune `yolov8n.pt` (nano model — fast inference, small file size).
- **100 epochs** — good balance of accuracy vs time
- **640px images** — standard YOLO resolution
- **~20-30 minutes** on Colab T4 GPU

In [ ]:
from ultralytics import YOLO

# Load the base YOLOv8 nano model (pretrained on COCO)
model = YOLO('yolov8n.pt')

# Train on shuttlecock dataset
results = model.train(
    data=os.path.join(dataset_location, 'data.yaml'),
    epochs=100,           # 100 epochs — good accuracy
    imgsz=640,            # Standard resolution
    batch=16,             # Batch size (fits in T4 GPU memory)
    name='shuttlecock',   # Run name
    patience=20,          # Early stop if no improvement for 20 epochs
    save=True,            # Save checkpoints
    plots=True,           # Generate training plots
    verbose=True,
    device=0,             # Use GPU
)

print("\n" + "="*60)
print("✅ TRAINING COMPLETE!")
print("="*60)

## Step 6: 📊 Evaluate the Model

In [ ]:
# Load the best model from training
import glob

best_model_path = glob.glob('runs/detect/shuttlecock*/weights/best.pt')
if best_model_path:
    best_path = sorted(best_model_path)[-1]  # Get the latest run
    print(f"📦 Best model: {best_path}")
    print(f"📏 File size: {os.path.getsize(best_path) / 1e6:.1f} MB")
    
    # Load and validate
    best_model = YOLO(best_path)
    metrics = best_model.val()
    
    print(f"\n{'='*60}")
    print(f"📊 VALIDATION RESULTS")
    print(f"{'='*60}")
    print(f"  Precision: {metrics.box.mp:.3f}")
    print(f"  Recall:    {metrics.box.mr:.3f}")
    print(f"  mAP@50:    {metrics.box.map50:.3f}")
    print(f"  mAP@50-95: {metrics.box.map:.3f}")
else:
    print("❌ No trained model found. Check training output above.")

## Step 7: 📊 View Training Curves

In [ ]:
from IPython.display import Image, display
import glob

# Show training results
result_imgs = glob.glob('runs/detect/shuttlecock*/results.png')
if result_imgs:
    display(Image(filename=sorted(result_imgs)[-1], width=900))

# Show confusion matrix
confusion_imgs = glob.glob('runs/detect/shuttlecock*/confusion_matrix.png')
if confusion_imgs:
    display(Image(filename=sorted(confusion_imgs)[-1], width=600))

# Show sample predictions
pred_imgs = glob.glob('runs/detect/shuttlecock*/val_batch0_pred.jpg')
if pred_imgs:
    print("\n📸 Sample predictions on validation set:")
    display(Image(filename=sorted(pred_imgs)[-1], width=800))

## Step 8: 💾 Download the Trained Model

The model file is only **~6MB**. Download it and place it in your project folder:
```
major project - Copy/
  └── shuttlecock_best.pt    ← Put it here!
```

In [ ]:
import shutil
from google.colab import files

# Copy best.pt to a friendly name
best_model_path = sorted(glob.glob('runs/detect/shuttlecock*/weights/best.pt'))[-1]
output_name = 'shuttlecock_best.pt'
shutil.copy2(best_model_path, output_name)

print(f"✅ Model saved as: {output_name}")
print(f"📏 Size: {os.path.getsize(output_name) / 1e6:.1f} MB")
print(f"\n🔽 Downloading to your computer...")

# Auto-download to user's computer
files.download(output_name)

print(f"\n{'='*60}")
print(f"🎉 ALL DONE!")
print(f"{'='*60}")
print(f"")
print(f"Next steps:")
print(f"1. Place 'shuttlecock_best.pt' in your project root folder")
print(f"   (same folder as app.py)")
print(f"2. The system will automatically detect and use it!")
print(f"{'='*60}")